In [2]:
# python libraries imports
import pandas as pd
import numpy as np
import io
import zipfile
import requests
import matplotlib.pyplot as plt

# customed local functions imports
import sys
sys.path.append("./scripts")

from utils import *

#### **Dataset load and details**

In [3]:
url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

r = requests.get(url)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))

ratings = pd.read_csv(z.open("ml-1m/ratings.dat"),sep="::",engine="python",names=["user", "item", "rating", "timestamp"])

In [4]:
print("============ Movielens 1M Dataset ===========")
print(f"#Ratings: {ratings["rating"].count()} | #Users: {ratings['user'].nunique()} | #Items: {ratings['item'].nunique()} ")

============ Movielens 1M Dataset ===========
#Ratings: 1000209 | #Users: 6040 | #Items: 3706 


In [5]:
# Normalize dividing by the max rating
max_rate = ratings["rating"].max()
ratings["rating"] = ratings["rating"]/max_rate

# drop timestamp
ratings = ratings[["user", "item","rating"]]

**Effectiveness**

In [6]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]

# Compute once, it doesn't depend on the variant
aa = average_ranking(ratings)

for distance in distances:
    for aggregate in aggregates:
        rep = reputation_ranking(ratings, distance, aggregate, lam=0.2)
        tau = kendall(rep, aa)

        results.append({
            "algorithm": f"{distance.upper()}-{aggregate.upper()}",
            "kendall_tau": tau,
        })

df_clean = pd.DataFrame(results)
display(df_clean)

,algorithm,kendall_tau
0,L1-AVG,0.997914
1,L1-MAX,0.988006
2,L1-MIN,0.998897
3,L2-AVG,0.998880
4,L2-MAX,0.993124
5,L2-MIN,0.999178


**Robustness**

In [9]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]
spam_ratios = [0.1, 0.3, 0.5]

# Noisy datasets: generated once per ratio, shared by all algorithms
noisy_datasets = {p: add_spammers(ratings, p) for p in spam_ratios}

# Baseline (AA): original vs noisy
aa_orig = average_ranking(ratings)
for p, noisy in noisy_datasets.items():
    results.append({
        "algorithm": "AA",
        "spam_ratio": p,
        "kendall_tau": kendall(aa_orig, average_ranking(noisy)),
    })

# Reputation-based algorithms: original vs noisy
for distance in distances:
    for aggregate in aggregates:
        rep_orig = reputation_ranking(ratings, distance, aggregate, lam=0.2)

        for p, noisy in noisy_datasets.items():
            rep_noisy = reputation_ranking(noisy, distance, aggregate, lam=0.2)

            results.append({
                "algorithm": f"{distance.upper()}-{aggregate.upper()}",
                "spam_ratio": p,
                "kendall_tau": kendall(rep_orig, rep_noisy),
            })

df_robust = pd.DataFrame(results)

# Table: one row per algorithm, one column per spam ratio
display(df_robust.pivot(index="algorithm", columns="spam_ratio", values="kendall_tau"))

spam_ratio,0.1,0.3,0.5
algorithm,,,
AA,0.748623,0.675111,0.646157
L1-AVG,0.752840,0.679827,0.650877
L1-MAX,0.749800,0.675404,0.645803
L1-MIN,0.747680,0.672384,0.641831
L2-AVG,0.749454,0.676174,0.647161
L2-MAX,0.748752,0.675393,0.646023
L2-MIN,0.748126,0.674634,0.645468
